In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

# ---------- Config ----------
CATALOG  = "oil_gas_demo"
SRC      = "/Volumes/oil_gas_demo/raw/raw_volume/supply_chain_dataset.csv"   
TGT      = f"{CATALOG}.bronze.supply_chain_transactions"

# ---------- Explicit schema: sab kuch STRING (bronze = as-is) ----------
cols = ["transaction_id","transaction_date","transaction_year","transaction_quarter",
        "transaction_month","product_name","product_category","quantity_unit",
        "supplier_name","supplier_country","supplier_reliability_score","refinery_name",
        "destination_city","transportation_mode","ordered_quantity","demand_quantity",
        "available_inventory","unit_price_usd","product_cost_usd","transportation_cost_usd",
        "total_cost_usd","expected_lead_time_days","actual_lead_time_days","delay_days",
        "is_delayed","is_stockout","quality_status","quality_score","disruption_type",
        "delivery_status","ingestion_timestamp","source_system","operation_type"]
schema = StructType([StructField(c, StringType(), True) for c in cols])

# ---------- Read ----------
df = (spark.read
      .option("header", True)
      .option("multiLine", True)
      .option("rescuedDataColumn", "_rescued_data")   # extra/bad columns yahan aa jayenge
      .schema(schema)
      .csv(SRC))

# ---------- Date column (as-is data ke saath, sirf ek naya column) ----------
df = df.withColumn(
    "txn_date",
    F.coalesce(
        F.expr("try_to_date(transaction_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(transaction_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(transaction_date, 'MM/dd/yyyy')"),
        F.expr("try_to_date(transaction_date, 'dd/MM/yyyy')"),
        F.to_date(F.expr("try_to_timestamp(transaction_date)")),   # timestamp format ho to
    )
)

# ---------- Audit columns ----------
df = (df.withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path")))

# ---------- Write ----------
(df.write.format("delta").mode("overwrite")
   .option("overwriteSchema", True)
   .saveAsTable(TGT))

spark.sql(f"COMMENT ON TABLE {TGT} IS 'Bronze: raw supply chain CSV as-is (all strings) + txn_date + audit columns'")

# ---------- Quick checks ----------
b = spark.table(TGT)
print("rows          :", b.count())
print("null txn_date :", b.filter("txn_date IS NULL").count())
print("rescued rows  :", b.filter("_rescued_data IS NOT NULL").count())
display(b.limit(10))